# 01 — Data Preparation and Tokenization

Arabic→English low-resource MT data + SentencePiece BPE. The raw download, cleaning,
normalization and BPE *training* were done once in the original development notebooks (now
in `notebooks/archive_original/`). **This notebook loads and verifies the prepared artifacts**
— it does not re-download data or retrain tokenizers.

Local step IDs: `DATA-01` … `DATA-05`.

## Setup

In [1]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
import pandas as pd
import sentencepiece as spm

TOK, VOC, MAX_LEN = 'Data/tokenized', 'Data/vocab', 80
def read_lines(p):
    return open(p, encoding='utf-8').read().splitlines()

## DATA-01 — Dataset and preprocessing summary
Source: **IWSLT 2017 Arabic-English** (TED talks). Raw train ~231,713 pairs, sampled to a
~50k low-resource subset. Preprocessing applied during the original prep:
- **Arabic:** light normalization — alef unification (أ/إ/آ → ا) and alef-maqsura → ya (ى → ي).
- **English:** lowercasing + whitespace normalization.
- Filtering: empty, over-length, badly-aligned and duplicate pairs removed.

The prepared parallel files (already tokenized to BPE) live under `Data/tokenized/`.

In [2]:
splits = ['train', 'validation', 'test']
counts = {s: (len(read_lines(f'{TOK}/{s}.ar.bpe')), len(read_lines(f'{TOK}/{s}.en.bpe'))) for s in splits}
pd.DataFrame([{'split': s, 'ar_lines': a, 'en_lines': e, 'aligned': a == e} for s, (a, e) in counts.items()])

,split,ar_lines,en_lines,aligned
0,train,50000,50000,True
1,validation,888,888,True
2,test,8567,8567,True


## DATA-02 — Tokenizers, vocabulary and special tokens
Separate SentencePiece BPE models for Arabic and English (subword units suit Arabic morphology).

In [3]:
sp_ar = spm.SentencePieceProcessor(model_file=f'{VOC}/sp_ar.model')
sp_en = spm.SentencePieceProcessor(model_file=f'{VOC}/sp_en.model')
pd.DataFrame([
    {'lang': 'ar', 'model': 'sp_ar.model', 'vocab_size': sp_ar.get_piece_size(),
     'pad': sp_ar.pad_id(), 'unk': sp_ar.unk_id(), 'bos': sp_ar.bos_id(), 'eos': sp_ar.eos_id()},
    {'lang': 'en', 'model': 'sp_en.model', 'vocab_size': sp_en.get_piece_size(),
     'pad': sp_en.pad_id(), 'unk': sp_en.unk_id(), 'bos': sp_en.bos_id(), 'eos': sp_en.eos_id()},
])

,lang,model,vocab_size,pad,unk,bos,eos
0,ar,sp_ar.model,8000,0,1,2,3
1,en,sp_en.model,8000,0,1,2,3


## DATA-03 — Max-length filtering
Pairs with > 80 BPE tokens on either side are dropped (CPU cost + padding); same filter used by training and evaluation.

In [4]:
rows = []
for s in splits:
    ar, en = read_lines(f'{TOK}/{s}.ar.bpe'), read_lines(f'{TOK}/{s}.en.bpe')
    kept = sum(1 for a, e in zip(ar, en) if len(a.split()) <= MAX_LEN and len(e.split()) <= MAX_LEN)
    rows.append({'split': s, 'total': len(ar), 'kept': kept, 'filtered': len(ar) - kept, 'max_len': MAX_LEN})
pd.DataFrame(rows)

,split,total,kept,filtered,max_len
0,train,50000,49441,559,80
1,validation,888,869,19,80
2,test,8567,8491,76,80


## DATA-04 — Dataset statistics
Length statistics and line-count validation saved during preparation.

In [5]:
for name in ['length_statistics.csv', 'line_count_validation.csv']:
    p = f'outputs/tables/{name}'
    if os.path.exists(p):
        print(name)
        display(pd.read_csv(p))

length_statistics.csv


,split,language,min,mean,median,p95,max
0,train,ar,1,21.64,17.0,51.00,189
1,train,en,2,23.24,19.0,54.00,183
2,validation,ar,1,23.64,19.0,55.65,112
3,validation,en,3,26.06,21.0,64.00,121
4,test,ar,1,20.84,17.0,49.00,128
5,test,en,2,22.27,19.0,51.00,121


line_count_validation.csv


,split,arabic_lines,english_lines,line_count_match,empty_arabic,empty_english
0,train,50000,50000,True,0,0
1,validation,888,888,True,0,0
2,test,8567,8567,True,0,0


## DATA-05 — Tokenizer sanity check (round-trip + coverage)
Decode a few `.bpe` lines back to text and re-encode; check that committed tokens are in the
vocabulary. (A small fraction of rare Arabic tokens map to `<unk>` — a documented minor
limitation; see the BPE verification in the original notebook.)

In [6]:
rows = []
for lang, sp in [('ar', sp_ar), ('en', sp_en)]:
    for i, line in enumerate(read_lines(f'{TOK}/test.{lang}.bpe')[:3]):
        pieces = line.split()
        text = sp.decode(pieces)
        rows.append({'lang': lang, 'idx': i, 'n_pieces': len(pieces),
                     'roundtrip_match': sp.encode(text, out_type=str) == pieces, 'text': text[:60]})
pd.DataFrame(rows)[['lang', 'idx', 'n_pieces', 'roundtrip_match', 'text']]

,lang,idx,n_pieces,roundtrip_match,text
0,ar,0,28,True,قبل عدة سنوات، هنا في تيد، قدّم بيتر سكيلمان م...
1,ar,1,53,True,والفكرة غاية في البساطة. فريق مكوّن من اربعة ي...
2,ar,2,11,True,يجب ان تكون المارش مالو علي القمة.
3,en,0,22,True,"several years ago here at ted, peter skillman ..."
4,en,1,48,True,and the idea's pretty simple: teams of four ha...
5,en,2,11,True,the marshmallow has to be on top.


In [7]:
rows = []
for lang, sp in [('ar', sp_ar), ('en', sp_en)]:
    vocab = {sp.id_to_piece(i) for i in range(sp.get_piece_size())}
    toks = [t for line in read_lines(f'{TOK}/test.{lang}.bpe') for t in line.split()]
    oov = sum(1 for t in toks if t not in vocab)
    rows.append({'lang': lang, 'test_tokens': len(toks), 'not_in_vocab': oov, 'pct': round(100 * oov / len(toks), 3)})
pd.DataFrame(rows)

,lang,test_tokens,not_in_vocab,pct
0,ar,178508,332,0.186
1,en,190795,3,0.002


## Summary
Prepared data (49,441 / 869 / 8,491 after filtering) and the two 8k SentencePiece tokenizers
load and verify correctly. The dictionary baseline is in notebook 02; Transformer training and
ablations in notebook 03.